In [ ]:
from pathlib import Path
import pandas as pd

# Ruta raíz del proyecto
BASE = Path.cwd()

# Si el notebook toma como carpeta actual 06_DEMOGRAFIA_SERVICIOS_ALIMENTOS,
# regresamos un nivel a DENUE_HISTORICOS
if not (BASE / "01_MORELIA_LIMPIO").exists():
    BASE = BASE.parent

CARPETA_MORELIA = BASE / "01_MORELIA_LIMPIO"

archivos = sorted(CARPETA_MORELIA.glob("DENUE_*_MORELIA.csv"))

print("Carpeta de trabajo:", BASE)
print("Número de cortes encontrados:", len(archivos))

for archivo in archivos:
    print(archivo.name)

In [2]:
# Auditoría de estructura de los 16 cortes

resumen = []
columnas_por_anio = {}

for archivo in archivos:
    anio = archivo.stem.split("_")[1]

    df = pd.read_csv(
        archivo,
        dtype=str,
        low_memory=False
    )

    columnas_por_anio[anio] = list(df.columns)

    resumen.append({
        "Año": anio,
        "Registros_Morelia": len(df),
        "Número_columnas": len(df.columns)
    })

resumen_df = pd.DataFrame(resumen)

display(resumen_df)

print("\nCOLUMNAS DEL PRIMER CORTE (2010):")
print(columnas_por_anio["2010"])

print("\nCOLUMNAS DEL ÚLTIMO CORTE (2026):")
print(columnas_por_anio["2026"])


,Año,Registros_Morelia,Número_columnas
0,2010,41342,24
1,2011,42004,30
2,2012,42251,47
3,2013,42672,45
4,2015,44814,41
5,2016,45511,42
6,2017,45715,42
7,2018,45900,42
8,2019,47934,42
9,2020,48981,42



COLUMNAS DEL PRIMER CORTE (2010):
['Nombre de la unidad económica', 'Razón social', 'Código de la clase de actividad', 'Nombre de la clase de actividad', 'Personal ocupado (estrato)', 'Calle, avenida, andador, carretera, manzana u otro', 'Número exterior o km', 'Edificio, piso o nivel', 'Número o letra interior', 'Colonia, fraccionamiento, unidad habitacional o barrio', 'Corredor industrial, centro comercial o mercado público', 'Número de local', 'Código postal', 'Entidad federativa', 'Municipio', 'Localidad', 'Área geoestadística básica', 'Manzana', 'Número de teléfono', 'Correo electrónico', 'Sitio en Internet', 'Tipo de unidad económica', 'Latitud', 'Longitud']

COLUMNAS DEL ÚLTIMO CORTE (2026):
['id', 'clee', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu', 'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1', 'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext', 'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int', 'tipo_asent', 'nomb_asen

In [3]:
import unicodedata

def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return texto

palabras_clave = [
    "actividad",
    "establecimiento",
    "unidad economica",
    "id",
    "clee",
    "latitud",
    "longitud",
    "fecha"
]

auditoria_columnas = []

for archivo in archivos:
    anio = archivo.stem.split("_")[1]

    # Solo leer encabezados
    columnas = pd.read_csv(
        archivo,
        nrows=0
    ).columns.tolist()

    relevantes = []

    for columna in columnas:
        columna_norm = normalizar_texto(columna)

        if any(palabra in columna_norm for palabra in palabras_clave):
            relevantes.append(columna)

    auditoria_columnas.append({
        "Año": anio,
        "Columnas relevantes": " | ".join(relevantes)
    })

auditoria_columnas_df = pd.DataFrame(auditoria_columnas)

display(auditoria_columnas_df)

,Año,Columnas relevantes
0,2010,Nombre de la unidad económica | Código de la c...
1,2011,Nombre de la unidad económica | Código de la c...
2,2012,Clave entidad | Entidad federativa | Clave loc...
3,2013,Clave entidad | Entidad federativa | Clave loc...
4,2015,ID | Nombre de la Unidad Económica | Código de...
5,2016,id | entidad | localidad | latitud | longitud ...
6,2017,id | entidad | localidad | latitud | longitud ...
7,2018,id | entidad | localidad | latitud | longitud ...
8,2019,id | entidad | localidad | latitud | longitud ...
9,2020,id | entidad | localidad | latitud | longitud ...


In [4]:
# Auditoría exacta de variables necesarias para el análisis longitudinal

pd.set_option("display.max_colwidth", None)

def normalizar_columna(texto):
    texto = str(texto).strip().lower()
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return texto


# Nombres equivalentes que pueden aparecer según la edición DENUE
equivalencias = {
    "ID": [
        "id"
    ],
    "CLEE": [
        "clee"
    ],
    "Nombre_establecimiento": [
        "nombre de la unidad economica",
        "nom_estab"
    ],
    "Razon_social": [
        "razon social",
        "raz_social"
    ],
    "Codigo_SCIAN": [
        "codigo de la clase de actividad",
        "codigo_act"
    ],
    "Nombre_actividad": [
        "nombre de la clase de actividad",
        "nombre_act"
    ],
    "Latitud": [
        "latitud"
    ],
    "Longitud": [
        "longitud"
    ],
    "Fecha_alta": [
        "fecha_alta",
        "fecha de alta"
    ]
}


auditoria = []

for archivo in archivos:
    anio = archivo.stem.split("_")[1]

    columnas_originales = pd.read_csv(
        archivo,
        nrows=0
    ).columns.tolist()

    columnas_normalizadas = {
        normalizar_columna(col): col
        for col in columnas_originales
    }

    fila = {"Año": anio}

    for variable, posibles_nombres in equivalencias.items():

        encontrada = None

        for nombre in posibles_nombres:
            if nombre in columnas_normalizadas:
                encontrada = columnas_normalizadas[nombre]
                break

        fila[variable] = encontrada if encontrada else "NO DISPONIBLE"

    auditoria.append(fila)


auditoria_variables = pd.DataFrame(auditoria)

display(auditoria_variables)

,Año,ID,CLEE,Nombre_establecimiento,Razon_social,Codigo_SCIAN,Nombre_actividad,Latitud,Longitud,Fecha_alta
0,2010,NO DISPONIBLE,NO DISPONIBLE,Nombre de la unidad económica,Razón social,Código de la clase de actividad,Nombre de la clase de actividad,Latitud,Longitud,NO DISPONIBLE
1,2011,NO DISPONIBLE,NO DISPONIBLE,Nombre de la unidad económica,Razón social,Código de la clase de actividad,Nombre de la clase de actividad,Latitud,Longitud,NO DISPONIBLE
2,2012,NO DISPONIBLE,NO DISPONIBLE,Nombre de la Unidad Económica,Razón social,NO DISPONIBLE,NO DISPONIBLE,Latitud,Longitud,NO DISPONIBLE
3,2013,NO DISPONIBLE,NO DISPONIBLE,Nombre de la Unidad Económica,Razón social,NO DISPONIBLE,NO DISPONIBLE,Latitud,Longitud,NO DISPONIBLE
4,2015,ID,NO DISPONIBLE,Nombre de la Unidad Económica,Razón social,NO DISPONIBLE,NO DISPONIBLE,Latitud,Longitud,NO DISPONIBLE
5,2016,id,NO DISPONIBLE,nom_estab,raz_social,codigo_act,nombre_act,latitud,longitud,fecha_alta
6,2017,id,NO DISPONIBLE,nom_estab,raz_social,codigo_act,nombre_act,latitud,longitud,fecha_alta
7,2018,id,NO DISPONIBLE,nom_estab,raz_social,codigo_act,nombre_act,latitud,longitud,fecha_alta
8,2019,id,NO DISPONIBLE,nom_estab,raz_social,codigo_act,nombre_act,latitud,longitud,fecha_alta
9,2020,id,NO DISPONIBLE,nom_estab,raz_social,codigo_act,nombre_act,latitud,longitud,fecha_alta


In [5]:
# Revisar encabezados completos de los años con aparente ausencia de SCIAN

for anio_buscar in ["2012", "2013", "2015"]:

    archivo = [
        a for a in archivos
        if a.stem.split("_")[1] == anio_buscar
    ][0]

    columnas = pd.read_csv(
        archivo,
        nrows=0
    ).columns.tolist()

    print("\n" + "=" * 70)
    print(f"COLUMNAS DEL CORTE {anio_buscar}")
    print("=" * 70)

    for i, columna in enumerate(columnas, start=1):
        print(f"{i:02d}. {columna}")


COLUMNAS DEL CORTE 2012
01. Llave DENUE
02. NIC
03. NOP
04. Clave entidad
05. Entidad federativa
06. Clave municipio
07. Municipio
08. Clave localidad
09. Localidad
10. Área geoestadística básica 
11. Manzana
12. Nombre de la Unidad Económica
13. Razón social
14. Tipo de vialidad
15. Nombre de la vialidad
16. Tipo entre vialidad 1
17. Nombre entre vialidad 1
18. Tipo entre vialidad 2
19. Nombre entre vialidad 2
20. Tipo vialidad posterior
21. Nombre vialidad posterior
22. Número exterior o kilómetro
23. Edificio, piso o nivel
24. Número o letra interior
25. Tipo de asentamiento humano
26. Nombre de asentamiento humano
27. Código Postal
28. Número de teléfono 1
29. Extensión 1
30. Número de teléfono 2
31. Extensión 2
32. Número de FAX
33. Código de la clase de actividad SCIAN
34. Nombre de clase de la actividad
35. Corredor industrial, centro comercial o mercado público
36. Número de local
37. Correo electrónico 1
38. Sitio en Internet
39. Correo electrónico 2
40. Tipo de establecimien

In [7]:
from pathlib import Path
import pandas as pd
import unicodedata

# ============================================================
# 1. LOCALIZAR LOS 16 ARCHIVOS DE MORELIA
# ============================================================

BASE = Path.cwd()

if not (BASE / "01_MORELIA_LIMPIO").exists():
    BASE = BASE.parent

CARPETA_MORELIA = BASE / "01_MORELIA_LIMPIO"

archivos = sorted(
    CARPETA_MORELIA.glob("DENUE_*_MORELIA.csv")
)

print("Cortes encontrados:", len(archivos))


# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def normalizar_columna(texto):
    texto = str(texto).strip().lower()

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    return texto


def buscar_columna(columnas, posibles_nombres):

    mapa = {
        normalizar_columna(col): col
        for col in columnas
    }

    for nombre in posibles_nombres:

        nombre_norm = normalizar_columna(nombre)

        if nombre_norm in mapa:
            return mapa[nombre_norm]

    return None


# ============================================================
# 3. INVENTARIO DE CLASES SCIAN 722
# ============================================================

inventario_722 = []

for archivo in archivos:

    anio = archivo.stem.split("_")[1]

    df = pd.read_csv(
        archivo,
        dtype=str,
        low_memory=False
    )

    col_codigo = buscar_columna(
        df.columns,
        [
            "Código de la clase de actividad",
            "Código de la clase de actividad SCIAN",
            "codigo_act"
        ]
    )

    col_nombre = buscar_columna(
        df.columns,
        [
            "Nombre de la clase de actividad",
            "Nombre de clase de la actividad",
            "nombre_act"
        ]
    )

    if col_codigo is None:
        print(f"ERROR: no se encontró SCIAN en {anio}")
        continue

    codigo_limpio = (
        df[col_codigo]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    df_722 = df[
        codigo_limpio.str.startswith("722", na=False)
    ].copy()

    df_722["codigo_scian"] = codigo_limpio.loc[df_722.index]

    if col_nombre is not None:
        df_722["nombre_actividad"] = df_722[col_nombre]
    else:
        df_722["nombre_actividad"] = pd.NA

    conteo = (
        df_722
        .groupby(
            ["codigo_scian", "nombre_actividad"],
            dropna=False
        )
        .size()
        .reset_index(name="establecimientos")
    )

    conteo.insert(0, "anio", int(anio))

    inventario_722.append(conteo)


inventario_722_df = pd.concat(
    inventario_722,
    ignore_index=True
)

inventario_722_df = inventario_722_df.sort_values(
    ["anio", "codigo_scian"]
).reset_index(drop=True)

display(inventario_722_df)

Cortes encontrados: 16


,anio,codigo_scian,nombre_actividad,establecimientos
0,2010,722110,RESTAURANTES CON SERVICIO COMPLETO,217
1,2010,722211,RESTAURANTES DE AUTOSERVICIO,150
2,2010,722212,RESTAURANTES DE COMIDA PARA LLEVAR,1975
3,2010,722219,OTROS RESTAURANTES CON SERVICIO LIMITADO,1707
4,2010,722320,SERVICIOS DE PREPARACION DE ALIMENTOS PARA OCASIONES ESPECIALES,8
...,...,...,...,...
183,2026,722515,"Cafeterías, fuentes de sodas, neverías, refresquerías y similares",845
184,2026,722516,Restaurantes de autoservicio,144
185,2026,722517,"Restaurantes con servicio de preparación de pizzas, hamburguesas, hot dogs y pollos rostizados para llevar",762
186,2026,722518,Restaurantes que preparan otro tipo de alimentos para llevar,545


In [8]:
# ============================================================
# RESUMEN DEL SUBSECTOR 722 POR CORTE
# ============================================================

resumen_722 = (
    inventario_722_df
    .groupby("anio")
    .agg(
        clases_scian_722=("codigo_scian", "nunique"),
        establecimientos_722=("establecimientos", "sum")
    )
    .reset_index()
)

display(resumen_722)

,anio,clases_scian_722,establecimientos_722
0,2010,7,4201
1,2011,7,4206
2,2012,8,4224
3,2013,8,4239
4,2015,13,5376
5,2016,13,5413
6,2017,12,5423
7,2018,12,5426
8,2019,14,6180
9,2020,13,6263


In [9]:
# ============================================================
# HISTORIA DE CADA CLASE SCIAN 722
# ============================================================

historia_codigos_722 = (
    inventario_722_df
    .groupby(["codigo_scian", "nombre_actividad"])
    .agg(
        primer_corte=("anio", "min"),
        ultimo_corte=("anio", "max"),
        numero_cortes=("anio", "nunique")
    )
    .reset_index()
    .sort_values(["primer_corte", "codigo_scian"])
)

display(historia_codigos_722)

,codigo_scian,nombre_actividad,primer_corte,ultimo_corte,numero_cortes
0,722110,RESTAURANTES CON SERVICIO COMPLETO,2010,2011,2
3,722211,RESTAURANTES DE AUTOSERVICIO,2010,2011,2
6,722212,RESTAURANTES DE COMIDA PARA LLEVAR,2010,2011,2
9,722219,OTROS RESTAURANTES CON SERVICIO LIMITADO,2010,2011,2
14,722320,SERVICIOS DE PREPARACION DE ALIMENTOS PARA OCASIONES ESPECIALES,2010,2011,2
18,722411,"CENTROS NOCTURNOS, DISCOTECAS Y SIMILARES",2010,2011,2
21,722412,"BARES, CANTINAS Y SIMILARES",2010,2011,2
2,722110,Servicios de alojamiento temporal y de preparación de alimentos y bebidas -Servicios de preparación de alimentos y bebidas -Restaurantes con servicio completo -Restaurantes con servicio completo,2012,2012,1
5,722211,"Servicios de alojamiento temporal y de preparación de alimentos y bebidas -Servicios de preparación de alimentos y bebidas -Restaurantes de autoservicio, comida para llevar y otros restaurantes con servicio limitado -Restaurantes de autoservicio, comida para llevar y otros restaurantes con servicio limitado",2012,2012,1
8,722212,"Servicios de alojamiento temporal y de preparación de alimentos y bebidas -Servicios de preparación de alimentos y bebidas -Restaurantes de autoservicio, comida para llevar y otros restaurantes con servicio limitado -Restaurantes de autoservicio, comida para llevar y otros restaurantes con servicio limitado",2012,2012,1


In [10]:
# ============================================================
# MATRIZ DE CORRESPONDENCIA OFICIAL SCIAN 2007 -> SCIAN 2013
# SUBSECTOR 722
# Fuente: INEGI, tabla comparativa SCIAN 2013 - SCIAN 2007
# ============================================================

from pathlib import Path
import pandas as pd

carpeta_fuentes = BASE / "06_DEMOGRAFIA_SERVICIOS_ALIMENTOS" / "00_FUENTES_SCIAN"
carpeta_fuentes.mkdir(parents=True, exist_ok=True)

correspondencias_2007_2013 = [
    # 722110 -> cinco clases nuevas
    ["722110", "Restaurantes con servicio completo",
     "722511", "Restaurantes con servicio de preparación de alimentos a la carta o de comida corrida",
     "p 1/5"],

    ["722110", "Restaurantes con servicio completo",
     "722512", "Restaurantes con servicio de preparación de pescados y mariscos",
     "p 2/5"],

    ["722110", "Restaurantes con servicio completo",
     "722513", "Restaurantes con servicio de preparación de antojitos",
     "p 3/5"],

    ["722110", "Restaurantes con servicio completo",
     "722514", "Restaurantes con servicio de preparación de tacos y tortas",
     "p 4/5"],

    ["722110", "Restaurantes con servicio completo",
     "722515", "Cafeterías, fuentes de sodas, neverías, refresquerías y similares",
     "p 5/5"],

    # 722219 -> seis clases nuevas
    ["722219", "Otros restaurantes con servicio limitado",
     "722511", "Restaurantes con servicio de preparación de alimentos a la carta o de comida corrida",
     "p 1/6"],

    ["722219", "Otros restaurantes con servicio limitado",
     "722512", "Restaurantes con servicio de preparación de pescados y mariscos",
     "p 2/6"],

    ["722219", "Otros restaurantes con servicio limitado",
     "722513", "Restaurantes con servicio de preparación de antojitos",
     "p 3/6"],

    ["722219", "Otros restaurantes con servicio limitado",
     "722514", "Restaurantes con servicio de preparación de tacos y tortas",
     "p 4/6"],

    ["722219", "Otros restaurantes con servicio limitado",
     "722515", "Cafeterías, fuentes de sodas, neverías, refresquerías y similares",
     "p 5/6"],

    ["722219", "Otros restaurantes con servicio limitado",
     "722519", "Servicios de preparación de otros alimentos para consumo inmediato",
     "p 6/6"],

    # Correspondencia directa
    ["722211", "Restaurantes de autoservicio",
     "722516", "Restaurantes de autoservicio",
     "Directa"],

    # División en dos clases
    ["722212", "Restaurantes de comida para llevar",
     "722517", "Restaurantes con servicio de preparación de pizzas, hamburguesas, hot dogs y pollos rostizados para llevar",
     "p 1/2"],

    ["722212", "Restaurantes de comida para llevar",
     "722518", "Restaurantes que preparan otro tipo de alimentos para llevar",
     "p 2/2"],
]

matriz_oficial_722 = pd.DataFrame(
    correspondencias_2007_2013,
    columns=[
        "SCIAN_2007",
        "Actividad_2007",
        "SCIAN_2013",
        "Actividad_2013",
        "Tipo_correspondencia"
    ]
)

display(matriz_oficial_722)

ruta_matriz = carpeta_fuentes / "MATRIZ_OFICIAL_SCIAN_722_2007_2013.csv"

matriz_oficial_722.to_csv(
    ruta_matriz,
    index=False,
    encoding="utf-8-sig"
)

print("Matriz guardada en:")
print(ruta_matriz)


,SCIAN_2007,Actividad_2007,SCIAN_2013,Actividad_2013,Tipo_correspondencia
0,722110,Restaurantes con servicio completo,722511,Restaurantes con servicio de preparación de alimentos a la carta o de comida corrida,p 1/5
1,722110,Restaurantes con servicio completo,722512,Restaurantes con servicio de preparación de pescados y mariscos,p 2/5
2,722110,Restaurantes con servicio completo,722513,Restaurantes con servicio de preparación de antojitos,p 3/5
3,722110,Restaurantes con servicio completo,722514,Restaurantes con servicio de preparación de tacos y tortas,p 4/5
4,722110,Restaurantes con servicio completo,722515,"Cafeterías, fuentes de sodas, neverías, refresquerías y similares",p 5/5
5,722219,Otros restaurantes con servicio limitado,722511,Restaurantes con servicio de preparación de alimentos a la carta o de comida corrida,p 1/6
6,722219,Otros restaurantes con servicio limitado,722512,Restaurantes con servicio de preparación de pescados y mariscos,p 2/6
7,722219,Otros restaurantes con servicio limitado,722513,Restaurantes con servicio de preparación de antojitos,p 3/6
8,722219,Otros restaurantes con servicio limitado,722514,Restaurantes con servicio de preparación de tacos y tortas,p 4/6
9,722219,Otros restaurantes con servicio limitado,722515,"Cafeterías, fuentes de sodas, neverías, refresquerías y similares",p 5/6


Matriz guardada en:
c:\Users\gilev\OneDrive\Desktop\DOCTORADO\DENUE_HISTORICOS\06_DEMOGRAFIA_SERVICIOS_ALIMENTOS\00_FUENTES_SCIAN\MATRIZ_OFICIAL_SCIAN_722_2007_2013.csv


In [11]:
# ============================================================
# FAMILIAS HOMOLOGADAS DEL SUBSECTOR 722
# PARA COMPARABILIDAD HISTÓRICA
# ============================================================

familias_722 = {

    # Familia 1:
    # antiguas clases 722110 + 722219
    # equivalentes conjuntamente a 722511-515 + 722519
    "722110": "F01",
    "722219": "F01",
    "722511": "F01",
    "722512": "F01",
    "722513": "F01",
    "722514": "F01",
    "722515": "F01",
    "722519": "F01",

    # Autoservicio
    "722211": "F02",
    "722516": "F02",

    # Comida para llevar
    "722212": "F03",
    "722517": "F03",
    "722518": "F03",

    # Comedores para empresas e instituciones
    "722310": "F04",

    # Preparación para ocasiones especiales
    "722320": "F05",

    # Unidades móviles
    "722330": "F06",

    # Centros nocturnos
    "722411": "F07",

    # Bares y cantinas
    "722412": "F08"
}


nombres_familias = {
    "F01": "Restaurantes y otros servicios de preparación para consumo inmediato",
    "F02": "Restaurantes de autoservicio",
    "F03": "Restaurantes y alimentos para llevar",
    "F04": "Servicios de comedor para empresas e instituciones",
    "F05": "Servicios de preparación de alimentos para ocasiones especiales",
    "F06": "Servicios de preparación de alimentos en unidades móviles",
    "F07": "Centros nocturnos, discotecas y similares",
    "F08": "Bares, cantinas y similares"
}


# Asegurar que SCIAN sea texto
inventario_722_df["codigo_scian"] = (
    inventario_722_df["codigo_scian"]
    .astype(str)
    .str.strip()
)

inventario_722_df["familia_homologada"] = (
    inventario_722_df["codigo_scian"]
    .map(familias_722)
)

inventario_722_df["nombre_familia"] = (
    inventario_722_df["familia_homologada"]
    .map(nombres_familias)
)

display(inventario_722_df.head(30))

,anio,codigo_scian,nombre_actividad,establecimientos,familia_homologada,nombre_familia
0,2010,722110,RESTAURANTES CON SERVICIO COMPLETO,217,F01,Restaurantes y otros servicios de preparación para consumo inmediato
1,2010,722211,RESTAURANTES DE AUTOSERVICIO,150,F02,Restaurantes de autoservicio
2,2010,722212,RESTAURANTES DE COMIDA PARA LLEVAR,1975,F03,Restaurantes y alimentos para llevar
3,2010,722219,OTROS RESTAURANTES CON SERVICIO LIMITADO,1707,F01,Restaurantes y otros servicios de preparación para consumo inmediato
4,2010,722320,SERVICIOS DE PREPARACION DE ALIMENTOS PARA OCASIONES ESPECIALES,8,F05,Servicios de preparación de alimentos para ocasiones especiales
5,2010,722411,"CENTROS NOCTURNOS, DISCOTECAS Y SIMILARES",20,F07,"Centros nocturnos, discotecas y similares"
6,2010,722412,"BARES, CANTINAS Y SIMILARES",124,F08,"Bares, cantinas y similares"
7,2011,722110,RESTAURANTES CON SERVICIO COMPLETO,222,F01,Restaurantes y otros servicios de preparación para consumo inmediato
8,2011,722211,RESTAURANTES DE AUTOSERVICIO,150,F02,Restaurantes de autoservicio
9,2011,722212,RESTAURANTES DE COMIDA PARA LLEVAR,1975,F03,Restaurantes y alimentos para llevar


In [12]:
# ============================================================
# CONTROL DE CALIDAD:
# ¿EXISTE ALGÚN CÓDIGO 722 SIN FAMILIA?
# ============================================================

sin_familia = (
    inventario_722_df[
        inventario_722_df["familia_homologada"].isna()
    ][
        ["codigo_scian", "nombre_actividad"]
    ]
    .drop_duplicates()
    .sort_values("codigo_scian")
)

print("Códigos sin familia homologada:", len(sin_familia))

display(sin_familia)

Códigos sin familia homologada: 0


,codigo_scian,nombre_actividad


In [13]:
# ============================================================
# PANEL HISTÓRICO POR FAMILIA HOMOLOGADA
# ============================================================

panel_familias_722 = (
    inventario_722_df
    .groupby(
        ["anio", "familia_homologada", "nombre_familia"],
        as_index=False
    )
    ["establecimientos"]
    .sum()
)

tabla_familias = (
    panel_familias_722
    .pivot(
        index=["familia_homologada", "nombre_familia"],
        columns="anio",
        values="establecimientos"
    )
    .fillna(0)
    .astype(int)
)

display(tabla_familias)

,anio,2010,2011,2012,2013,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
familia_homologada,nombre_familia,,,,,,,,,,,,,,,,
F01,Restaurantes y otros servicios de preparación para consumo inmediato,1924,1930,1943,1962,3945,3972,3977,3977,4340,4443,4418,4419,4438,5010,5127,5139
F02,Restaurantes de autoservicio,150,150,151,148,53,57,60,60,210,108,107,107,107,197,141,144
F03,Restaurantes y alimentos para llevar,1975,1975,1977,1971,1120,1122,1125,1128,1329,1416,1397,1396,1393,1227,1298,1307
F04,Servicios de comedor para empresas e instituciones,0,0,1,1,1,1,0,0,7,5,8,8,7,12,2,3
F05,Servicios de preparación de alimentos para ocasiones especiales,8,8,9,10,16,16,16,16,16,16,16,16,15,17,17,18
F06,Servicios de preparación de alimentos en unidades móviles,0,0,0,0,0,0,0,0,6,0,0,0,0,13,5,5
F07,"Centros nocturnos, discotecas y similares",20,19,19,19,12,12,12,12,16,17,17,17,17,11,11,11
F08,"Bares, cantinas y similares",124,124,124,128,229,233,233,233,256,258,258,258,262,261,269,274


In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# GUARDAR RESULTADOS DEL PRIMER AVANCE
# DEMOGRAFÍA DE NEGOCIOS - SUBSECTOR 722
# ============================================================

# Carpeta del estudio
CARPETA_ESTUDIO = BASE / "06_DEMOGRAFIA_SERVICIOS_ALIMENTOS"

# Carpeta para resultados
CARPETA_RESULTADOS = CARPETA_ESTUDIO / "RESULTADOS"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. INVENTARIO ORIGINAL DE CLASES SCIAN 722 POR CORTE
# ------------------------------------------------------------

ruta_inventario = CARPETA_RESULTADOS / "INVENTARIO_CLASES_SCIAN_722_POR_CORTE.csv"

inventario_722_df.to_csv(
    ruta_inventario,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 2. RESUMEN DEL SUBSECTOR 722 POR CORTE
# ------------------------------------------------------------

ruta_resumen = CARPETA_RESULTADOS / "RESUMEN_722_POR_CORTE.csv"

resumen_722.to_csv(
    ruta_resumen,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. GRUPOS DE COMPARABILIDAD SCIAN
# ------------------------------------------------------------

grupos_comparabilidad_722 = panel_familias_722.rename(
    columns={
        "familia_homologada": "grupo_comparabilidad_scian",
        "nombre_familia": "nombre_grupo"
    }
)

ruta_grupos = CARPETA_RESULTADOS / "GRUPOS_COMPARABILIDAD_SCIAN_722.csv"

grupos_comparabilidad_722.to_csv(
    ruta_grupos,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4. TABLA HORIZONTAL DE LOS 16 CORTES
# ------------------------------------------------------------

tabla_grupos_exportar = tabla_familias.reset_index().rename(
    columns={
        "familia_homologada": "grupo_comparabilidad_scian",
        "nombre_familia": "nombre_grupo"
    }
)

ruta_tabla = CARPETA_RESULTADOS / "TABLA_GRUPOS_COMPARABILIDAD_722_2010_2026.csv"

tabla_grupos_exportar.to_csv(
    ruta_tabla,
    index=False,
    encoding="utf-8-sig"
)


print("ARCHIVOS GENERADOS CORRECTAMENTE:")
print()
print(ruta_inventario)
print(ruta_resumen)
print(ruta_grupos)
print(ruta_tabla)

## Resultado del primer avance

Se delimitó el análisis al subsector SCIAN 722, correspondiente a los servicios de preparación de alimentos y bebidas, utilizando los 16 cortes históricos del DENUE disponibles para el municipio de Morelia.

La exploración mostró cambios en la clasificación de las clases de actividad a lo largo del periodo. Por ello, los códigos SCIAN específicos no fueron tratados como categorías directamente comparables entre todos los cortes.

Se conservaron los códigos y nombres originales de cada clase de actividad y, adicionalmente, se construyeron grupos de comparabilidad SCIAN para facilitar el análisis longitudinal de las categorías afectadas por cambios de clasificación.

Los grupos de comparabilidad tienen únicamente una función metodológica de seguimiento histórico y no sustituyen a las clases SCIAN originales.

En esta etapa todavía no se identifican nacimientos, muertes o supervivencia de establecimientos. El análisis realizado corresponde a la delimitación del universo, revisión de la estructura histórica del subsector 722 y preparación de la base para el posterior seguimiento longitudinal de establecimientos individuales.